In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# Assignment 2 Basic Config
# =========================

START_DATE = "2018-01-01"
END_DATE = "2025-12-31"

# yfinance 的 end 是 exclusive，所以这里用 2026-01-01 才能覆盖 2025-12-31 附近
DOWNLOAD_END = "2026-01-01"

OUTPUT_DIR = Path("assignment2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================
# Asset Groups
# =========================

growth_tech = ["AAPL", "MSFT", "NVDA", "META", "AMZN", "GOOGL", "TSLA"]

real_assets = ["XOM", "CVX", "FCX", "BHP", "GLD", "SLV"]

defensive_assets = ["TLT", "IEF", "XLU", "XLV", "IAU", "SHY", "UUP"]

default_assets = ["SPY", "QQQ", "IAU", "XLU", "XLV"]

market_symbols = ["SPY", "QQQ", "^GSPC", "^VIX"]

asset_groups = {
    "growth_tech": growth_tech,
    "real_assets": real_assets,
    "defensive": defensive_assets,
    "default": default_assets,
}

all_tickers = sorted(set(
    growth_tech 
    + real_assets 
    + defensive_assets 
    + default_assets 
    + market_symbols
))

print("Number of tickers:", len(all_tickers))
print(all_tickers)

# =========================
# Download Price Data
# =========================

raw = yf.download(
    all_tickers,
    start=START_DATE,
    end=DOWNLOAD_END,
    auto_adjust=True,
    progress=False,
    group_by="ticker",
    threads=True
)

# Convert yfinance output into a clean close-price matrix
close_dict = {}

for ticker in all_tickers:
    try:
        if isinstance(raw.columns, pd.MultiIndex):
            if ticker in raw.columns.get_level_values(0):
                close_dict[ticker] = raw[ticker]["Close"]
        else:
            # 如果只有一个 ticker 时才会进入这里
            close_dict[ticker] = raw["Close"]
    except Exception as e:
        print(f"Skip {ticker}: {e}")

close = pd.DataFrame(close_dict).sort_index()
close = close.dropna(how="all").ffill()

# Restrict to assignment backtest window
close = close.loc[(close.index >= START_DATE) & (close.index <= END_DATE)]

daily_ret = close.pct_change().fillna(0)

# Weekly close: use Friday weekly close
weekly_close = close.resample("W-FRI").last().dropna(how="all").ffill()
weekly_ret = weekly_close.pct_change().dropna(how="all")

print("Daily close shape:", close.shape)
print("Weekly close shape:", weekly_close.shape)
display(close.tail())